In [66]:
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

In [67]:
def model_name():
    return ChatOpenAI(model="gpt-4o-mini")

model = model_name()

In [80]:
class PostState(TypedDict):
    topic: str
    tweet: str
    evaluation: str      # "approved" or "needs_improvement"
    feedback: str
    iteration: int

In [81]:
class Evaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(
        description="Whether the post is good enough to publish or needs improvement"
    )
    feedback: str = Field(
        description="Specific feedback on what to improve, empty if approved"
    )

In [82]:
def generate(state: PostState) -> dict:
    topic = state["topic"]

    prompt = f"""
Write a short, engaging tweet about the following topic:

Topic: {topic}
"""

    response = model.invoke(prompt)
    return {"tweet": response.content, "iteration": 1}

In [83]:
def evaluate(state: PostState) -> dict:
    tweet = state["tweet"]
    iteration = state.get("iteration", 1)

    # Agar 3 se zyada iterations ho gayi to force approve
    if iteration >= 3:
        return {"evaluation": "approved", "feedback": "Max iterations reached, auto-approved."}

    structured_model = model.with_structured_output(Evaluation)

    prompt = f"""
Evaluate the following tweet for quality, clarity, and engagement.

Tweet:
{tweet}

If it is good enough to publish, mark it "approved".
If it needs work, mark it "needs_improvement" and give specific feedback.
"""

    result: Evaluation = structured_model.invoke(prompt)

    return {"evaluation": result.evaluation, "feedback": result.feedback}

In [84]:
def check_evaluation(state: PostState) -> Literal["approved", "needs_improvement"]:
    return state["evaluation"]

In [85]:
def optimize(state: PostState) -> dict:
    tweet = state["tweet"]
    feedback = state["feedback"]
    iteration = state.get("iteration", 1)

    prompt = f"""
Improve the following tweet based on this feedback.

Original Tweet:
{tweet}

Feedback:
{feedback}

Return only the improved post.
"""

    response = model.invoke(prompt)
    return {"tweet": response.content, "iteration": iteration + 1}

In [86]:
builder = StateGraph(PostState)

# Nodes
builder.add_node("generate", generate)
builder.add_node("evaluate", evaluate)
builder.add_node("optimize", optimize)

# Edges
builder.add_edge(START, "generate")
builder.add_edge("generate", "evaluate")

# Conditional Edge
builder.add_conditional_edges(
    "evaluate",
    check_evaluation,
    {
        "approved": END,
        "needs_improvement": "optimize"
    }
)

# Loop: from optimize to evaluate
builder.add_edge("optimize", "evaluate")

graph = builder.compile()

In [87]:
p1 = PostState()
p1

{}

In [75]:
import time

In [76]:
if __name__ == "__main__":
    initial_input = {"topic": input("Enter topic: ")}

    print("\n--- STREAMING WORKFLOW ---\n")

    final_state = {}
    for chunk in graph.stream(initial_input, stream_mode="updates"):
        for node_name, node_output in chunk.items():
            print(f"🔹 Node: {node_name}")
            print(f"   Output: {node_output}")
            print("-" * 50)
            final_state.update(node_output)
            time.sleep(1.0)

    print("\n--- FINAL TWEET ---")
    print(final_state.get("tweet"))
    print(f"\nEvaluation: {final_state.get('evaluation')}")
    print(f"Total Iterations: {final_state.get('iteration', 1)}")


--- STREAMING WORKFLOW ---

🔹 Node: generate
   Output: {'tweet': "🚀🤖 Exciting times ahead! AI isn't just a buzzword—it's a game changer! From transforming industries to enhancing our daily lives, the future is smart and full of possibilities. What role do you think AI will play in our world? 🌍✨ #AIRevolution #FutureTech", 'iteration': 1}
--------------------------------------------------
🔹 Node: evaluate
   Output: {'evaluation': 'approved', 'feedback': ''}
--------------------------------------------------

--- FINAL TWEET ---
🚀🤖 Exciting times ahead! AI isn't just a buzzword—it's a game changer! From transforming industries to enhancing our daily lives, the future is smart and full of possibilities. What role do you think AI will play in our world? 🌍✨ #AIRevolution #FutureTech

Evaluation: approved
Total Iterations: 1
